# PyInvest — Análise do Projeto e "Mode Expert"

> Documento gerado automaticamente: descrição do projeto, fluxo de tarefas, análise detalhada da funcionalidade **Modo Expert** (fluxo de dados), e identificação de potenciais bugs e recomendações.

---

## Sumário ✅

- Visão geral do projeto
- Estrutura do repositório
- Fluxo de trabalho / To-Do (tarefas priorizadas)
- Análise da funcionalidade **Modo Expert** 🔬 (UI, dados e comportamento esperado)
- Fluxo de dados detalhado (diagramas textuais)
- Problemas e potenciais bugs identificados ⚠️
- Recomendações de correção e testes 🔧

---

## 1. Visão geral do projeto

PyInvest é um simulador financeiro com foco em projeções e análise probabilística (Monte Carlo). Principais responsabilidades:

- UI (PySide6): janelas, widgets, diálogo de dados históricos e gráficos (plotly via HTML/`QWebEngineView`).
- Core (Python): motor Monte Carlo (`core/monte_carlo.py`), funções estatísticas (`core/statistics.py`), cálculo determinístico/estocástico.
- Persistência: projeto `.pyinv` (serialização em JSON), import/export de históricos.

---

## 2. Estrutura chave do repositório

- `main.py` — ponto de entrada
- `ui/` — interface do aplicativo
  - `window_modern.py` — principal janela, contém checkbox e opções do **Modo Expert**
  - `historical_dialog.py` — diálogo para gerenciar retornos históricos
  - `advanced_widgets.py` — `ProjectionChartExpert`, gráficos para Modo Expert
- `core/` — lógica do domínio
  - `monte_carlo.py` — Motor Monte Carlo (`MonteCarloEngine`, `MonteCarloWorker`)
  - `statistics.py` — funções estatísticas e métodos de simulação (`bootstrap_returns`, `normal_returns`, `t_student_returns`)

---

## 3. Fluxo de tarefas sugerido (alto nível) 📝

1. **Documentar** (este arquivo) — feito ✅
2. **Adicionar campos formais ao `MonteCarloInput`** para `expert_mode`, `simulation_method`, `historical_returns`, `random_seed`. (Alta prioridade)
3. **Integrar métodos de simulação do `statistics.py` no `MonteCarloEngine`**: suportar `bootstrap`, `normal`, `t_student` com sequências anuais de retorno. (Alta)
4. **Corrigir uso de RNG** para usar `numpy.random.Generator` em vez de `np.random.seed` (thread-safe & sem side effects). (Média)
5. **Adicionar testes automatizados** (unit tests) cobrindo cada método e integração do Modo Expert. (Média)
6. **Aprimorar conversões de unidades** (garantir consistência entre % e decimal nos pontos de entrada). (Baixa)

---

## 4. Modo Expert — descrição e comportamento esperado 🔬

### UI

- Checkbox: `self.check_expert_mode` — controla exibição de `self.expert_options`.
- Método de simulação: `self.combo_method` com opções:
  - `Bootstrap Histórico (Recomendado)`
  - `Distribuição Normal (Gaussiana)`
  - `Distribuição t-Student (Caudas Pesadas)`
- Botão `📊 Dados de Rendimento` abre `HistoricalReturnsDialog` para importar/editar retornos anuais (classe `HistoricalReturn` em `core/statistics.py`).

### Comportamento esperado

Ao ativar Modo Expert e disparar a simulação:

1. Se método = `bootstrap` → Deve reamostrar (com reposição) os retornos históricos para gerar uma matriz (n_sim x anos), ou seja, um vetor de retornos por ano para cada simulação.
2. Se método = `normal` ou `t_student` → Deve gerar matrizes (n_sim x anos) com amostras anuais usando parâmetros (média, std) derivados dos dados históricos ou valores fornecidos.
3. Essas matrizes anuais devem ser convertidas para taxas mensais (por ano) para efeito do cálculo mês-a-mês do saldo (monthly_rate = (1 + annual_return)^(1/12) - 1), e então aplicadas dinamicamente por cada ano nos 12 meses correspondentes.
4. O restante da pipeline (cálculo mensal com eventos extraordinários, agregação percentis e cenários representativos) deve consumir o resultado vetorizado (matriz `all_balances`) como atualmente já faz.

---

## 5. Fluxo de dados atual (observado) — resumo

1. Usuário ativa `Modo Expert` e seleciona método.
2. Usuário fornece histórico pelo `HistoricalReturnsDialog` (lista de `HistoricalReturn` com `return_rate` em decimal, ex: 0.1263 para 12.63%).
3. `window_modern._validate_inputs()` deriva `ParameterRange` para `rentabilidade_anual` (usa `avg_return = mean(historical_returns * 100)` **em %** para construir ranges).
4. `mc_input` (instância `MonteCarloInput`) é criado — **atributos 'expert_mode', 'historical_returns', 'simulation_method' são adicionados dinamicamente** em runtime.
5. `MonteCarloEngine.run()` é executado; porém, o código atualmente **não** consulta `mc_input.expert_mode` nem `simulation_method` nem `historical_returns` — em vez disso, usa `ParameterRange.sample()` para gerar um único valor anual (por simulação) e o aplica de forma constante para todos os anos. Resultado: técnicas Expert (bootstrap/normal/t-student) NÃO são aplicadas.

> Conclusão: o comportamento real não corresponde ao comportamento esperado do Modo Expert.

---

## 6. Problemas e potenciais bugs identificados ⚠️

1. **Modo Expert não é aplicado no motor** (crítico):
   - `window_modern` armazena `mc_input.simulation_method` e `mc_input.historical_returns`, mas `MonteCarloEngine` ignora essas propriedades. O motor só usa `ParameterRange.sample()` (um valor anual por simulação).

2. **Campos adicionados dinamicamente** ao `MonteCarloInput` (potencial fonte de erros silenciosos):
   - Ex.: `mc_input.expert_mode = True` e `mc_input.historical_returns = ...` são definidos em runtime, mas não constam dos campos dataclass — recomenda-se adicionar explicitamente tais campos ao dataclass.

3. **Unidades inconsistentes / possibilidades de confusão**:
   - Alguns trechos usam `return_rate * 100` para cálculos de média/std (em %), enquanto as funções `bootstrap_returns`/`normal_returns` esperam valores numéricos; é fácil confundir se os valores são `0.1263` (decimal) ou `12.63` (percentual). Documentar claramente e unificar unidades.

4. **Uso de RNG global (`np.random.seed`) dentro de funções** (`bootstrap_returns`, `normal_returns`, `t_student_returns`) pode introduzir efeitos colaterais e *race conditions* quando executado em threads (o `MonteCarloWorker` executa a simulação em uma thread). Melhor usar `numpy.random.Generator` com seed local.

5. **Bootstrap vs modelo atual**:
   - `bootstrap_returns` produz uma matriz anual (n_sim x n_years), enquanto o motor atual amostra um valor anual por simulação — isso não corresponde ao método de bootstrap esperado.

6. **Percentil/Índice nos cálculos de `calculate_percentiles`**:
   - Implementação usa índice inteiro (int(0.05 * n)) ao invés de `np.percentile`, o que simplifica mas pode produzir discretizações inesperadas em tamanhos pequenos.

7. **Potenciais problemas numéricos**:
   - Em `t_student_returns`, a escala usa `scale_factor = std_return * sqrt((df - 2) / df)` — é correto para reduzir a variância de `t` para `std_return`, mas para `df <= 2` o código cai no `else` (usa std_return) — recomenda-se validar `df > 2` e documentar comportamento.

---

## 7. Recomendações de correção e roadmap de implementação 🔧

Curto prazo (prioridade alta):

1. **Adicionar campos ao `MonteCarloInput`**:
   - `expert_mode: bool = False`
   - `simulation_method: str = 'bootstrap'`
   - `historical_returns: Optional[List[HistoricalReturn]] = None`
   - `random_seed: Optional[int] = None`

2. **Aprimorar `MonteCarloEngine.run()`** para suportar Modo Expert:
   - Se `inputs.expert_mode`:
     - Se `simulation_method == 'bootstrap'`: chamar `bootstrap_returns([r.return_rate for r in historical_returns], years, n_simulations, seed)` e receber matriz `n_sim x years` de retornos anuais.
     - Se `simulation_method == 'normal'` ou `t_student`: calcular `mean` e `std` a partir de dados históricos (ou parâmetros fornecidos) e chamar `normal_returns`/`t_student_returns` para obter matriz anual.
   - Converter matriz anual para `monthly_rates` (expandir cada ano para 12 meses), e executar simulação vetorizada mês-a-mês usando esses monthly_rates para cada simulação (vetor de taxas por mês para cada simulação).

3. **Evitar `np.random.seed` global** e usar `Generator`:
   - Ex.: `rng = np.random.default_rng(seed)` e usar `rng.choice` / `rng.normal` / `rng.standard_t`.

4. **Unificar as unidades**:
   - Decidir se os métodos de geração recebem `annual_return` em decimal (ex: 0.1263) ou percent (12.63). Documentar e transformar explicitamente nas interfaces.

Médio prazo:

- Adicionar testes unitários para:
  - `bootstrap_returns` com pequenas séries históricas
  - Integração `MonteCarloEngine` + Método Expert (validação de percentis e cenários representativos)
- Validar comportamento multi-thread (worker)

---

## 8. Tarefas recomendadas (lista prática) ✅

1. (Alta) Atualizar `core/monte_carlo.py` — adicionar campos no dataclass `MonteCarloInput`.
2. (Alta) Implementar integração `simulation_method` → `MonteCarloEngine`.
3. (Alta) Atualizar `core/statistics.py` para usar `Generator` em vez de `np.random.seed`.
4. (Média) Refatorar `window_modern._validate_inputs()` para **não** multiplicar por 100 onde for ambíguo (documentar conversões).
5. (Média) Adicionar testes unitários e de integração.
6. (Baixa) Melhorar mensagens de ajuda/tooltip no UI sobre o que cada método faz e requisitos (ex.: bootstrap precisa de >=2 anos).

---

## 9. Pontos de verificação / QA

- Testar com histórico curto (2 anos) e confirmar erros/avisos adequados
- Testar seed reproducibility com `random_seed` definido
- Verificar consistência entre valores exibidos na UI e os usados internamente (ex.: média calculada vs média aplicada nas simulações)

---

## 10. Conclusão

O Modo Expert tem UI e utilitários estatísticos implementados, mas **a integração está incompleta**: o motor Monte Carlo atual não consome os métodos de simulação do modo expert (bootstrap/normal/t-student) nem usa os retornos históricos por ano. Corrigir isso trará maior fidelidade às análises e permitirá que a aplicação entregue o comportamento esperado no Modo Expert.


---

> Gerado por análise automática — para prosseguir posso abrir PRs com:
> - mudanças no `MonteCarloInput` e `MonteCarloEngine` (implementação das amostragens por ano),
> - refatoração do uso de RNG,
> - testes unitários.

